<a href="https://colab.research.google.com/github/iniehil/prototype_customer_churn_risk/blob/main/prototype_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Title of the app
st.title("Who is likely to chrun - and when?")

In [ ]:
# Instructions of the app
st.markdown("Upload all required reports.")

In [7]:
# Upload Salesforce report
#sf_file = st.file_uploader("Upload the Salesforce Report")
df_sf = pd.read_csv('synthetic_salesforce.csv')

In [5]:
# Upload Omni report
#omni_file = st.file_uploader("Upload the Omni Report")
df_omni = pd.read_csv('synthetic_omni.csv')

In [6]:
# Upload Pendo report
#pendo_file = st.file_uploader("Upload the Pendo Report")
df_pendo = pd.read_csv('synthetic_pendo.csv')

In [8]:
# Upload Zendesk report
#zendesk_file = st.file_uploader("Upload the Zendesk Report")
df_zendesk = pd.read_csv('synthetic_zendesk.csv')

In [10]:
# Merge all input files on account ID
df = pd.merge(df_sf, df_omni, on='customer_id', how='outer')
df = pd.merge(df, df_pendo, on='customer_id', how='outer')
df = pd.merge(df, df_zendesk, on='customer_id', how='outer')

In [14]:
def generate_risk_score(df):
  if df['days_to_renewal'] <= 30 and df['usage_change_90d_pct'] <= -25.0:
    df['risk_score'] = 'High';
  elif df['days_to_renewal'] <= 90 and df['usage_change_90d_pct'] <= 5.0:
    df['risk_score'] = 'Medium';
  else:
    df['risk_score'] = 'Low'

  return df


In [ ]:
# Display dataframe
df = df.apply(generate_risk_score, axis=1)

config = {
    "customer_id": st.column_config.TextColumn("Customer ID", width="medium"),
    "customer_name": st.column_config.TextColumn("Customer Name", width="medium"),
    "renewal_date": st.column_config.DateColumn("Renewal Date", format="MMM, DD YYYY"),
    "days_to_renewal": st.column_config.NumberColumn("Days to Renewal"),
    "annual_revenue": st.column_config.NumberColumn("Annual Revenue ($)"),
    "risk_score": st.column_config.NumberColumn("Risk Score")
}

st.dataframe(data=df, width='stretch', column_config=config)